# Train Card Strength Model

**Objectives:**
- Prepare the dataset for training by splitting and scaling features.
- Train an late fusion model.
- Evaluate model accuracy using MAE and RMSE metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import copy

from embeddings.mtg_dataset import MTGDataset
from embeddings.mtg_fusion_model import MTGFusionModel

In [ ]:
INPUT_DATA_PATH = "../data/training/card_strength_input.parquet"
LABEL_DATA_PATH = "../data/training/card_strength_metric.xlsx"
OUTPUT_MODEL_PATH = "../models/card_strength_model.pth"
SEED = 1893
EPOCHS = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def train_torch_model(
    model, 
    train_loader, 
    val_loader, 
    optimizer, 
    criterion, 
    device, 
    epochs=50, 
    verbose=10
):
    """
    Reusable training loop that tracks and returns the best model based on validation loss.
    """
    best_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0
        for texts, tabs, labels in train_loader:
            texts, tabs, labels = texts.to(device), tabs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(texts, tabs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * texts.size(0)
        
        train_loss /= len(train_loader.dataset)
        
        # Validation phase
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for texts, tabs, labels in val_loader:
                texts, tabs, labels = texts.to(device), tabs.to(device), labels.to(device)
                outputs = model(texts, tabs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * texts.size(0)
        
        val_loss /= len(val_loader.dataset)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            
        if (epoch + 1) % verbose == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            
    # Load best weights before returning
    model.load_state_dict(best_model_wts)
    return model, history

In [ ]:
np.random.seed(SEED)
df_input = pd.read_parquet(INPUT_DATA_PATH)
df_label = pd.read_excel(LABEL_DATA_PATH)

cols_to_use = df_label.columns.difference(df_input.columns).tolist()
df = df_input.merge(df_label[cols_to_use + ["oracle_id"]], on="oracle_id", how="left")

# TODO (for me): investigate how log_power_score of nan is possible earlier in the workflow
print(f"Shape: {df.shape}")
# Remove rows with missing targets or infinite values
df = df.dropna(subset=["log_power_score"])
df = df[np.isfinite(df["log_power_score"])]

print(f"Shape: {df.shape}")
df.head()[df.columns[:8]]

## Data Filtering

Exclude most recent set (and every card that has been release since to avoid inaccuracies)

In [ ]:
# TODO (for later): from all sets find the most recent (that has been released)
# -> filter all cards from that set and cards that are released after that (like secret lairs)

## Feature Preparation

In [ ]:
embedding_cols = [col for col in df.columns if col.startswith('embed_')]
numerical_cols = [col for col in df.columns if col.startswith('num_')]
boolean_cols = [col for col in df.columns if col.startswith('bool_')]
embedding_features = df[embedding_cols].values
numerical_features = df[numerical_cols].values
boolean_features = df[boolean_cols].values

feature_cols = embedding_cols + numerical_cols + boolean_cols
target_col = 'log_power_score'

print(f"Number of embedding dimensions: {len(embedding_cols)}")
print(f"Number of numerical features: {len(numerical_cols)}")
print(f"  -> {", ".join(numerical_cols)}")
print(f"Number of boolean features: {len(boolean_cols)}")
print(f"  -> {", ".join(boolean_cols)}")

## Correlation Analysis

- Calculate Pearson correlation between features and the target power score.
- Plot the top positive and negative correlations to validate feature engineering.

In [ ]:
correlations = df[feature_cols + [target_col]].corr()[target_col].drop(target_col)
top_correlations = pd.concat([correlations.nsmallest(10), correlations.nlargest(10)]).sort_values()

plt.figure(figsize=(12, 8))
sns.barplot(x=top_correlations.values, y=top_correlations.index)
plt.title('Top Correlations with Power Score', fontsize=16)
plt.show()

## Data Splitting

- Split data into Training (60%), Validation (20%), and Test (20%) sets.
- Ensure stratification and reproducibility using the defined seed.

In [ ]:
text_embeddings = df[embedding_cols].values
tabular_features = df[numerical_cols + boolean_cols].fillna(-1).values

# TODO: maybe have to scale embedding features?
tabular_features = df[numerical_cols + boolean_cols].fillna(-1).values
scaler = StandardScaler()
tabular_features = scaler.fit_transform(tabular_features)

targets = df[target_col].values

X_text_train_val, X_text_test, X_tab_train_val, X_tab_test, y_train_val, y_test = train_test_split(
    text_embeddings, tabular_features, targets, test_size=0.20, random_state=SEED
)

X_text_train, X_text_val, X_tab_train, X_tab_val, y_train, y_val = train_test_split(
    X_text_train_val, X_tab_train_val, y_train_val, test_size=0.25, random_state=SEED
)

train_ds = MTGDataset(X_text_train, X_tab_train, y_train)
val_ds = MTGDataset(X_text_val, X_tab_val, y_val)
test_ds = MTGDataset(X_text_test, X_tab_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
test_loader = DataLoader(test_ds, batch_size=64)

print(f"Training data: {X_text_train.shape[0]} cards")
print(f"Validation data: {X_text_val.shape[0]} cards")
print(f"Test data: {X_text_test.shape[0]} cards")

## Create baselines

### Full XGBoost baseline

In [ ]:
X_train_xgb = np.hstack([X_text_train, X_tab_train])
X_val_xgb = np.hstack([X_text_val, X_tab_val])
X_test_xgb = np.hstack([X_text_test, X_tab_test])

xgb_reg = XGBRegressor(
    random_state=SEED, 
    device=DEVICE
)

xgb_reg.fit(
    X=X_train_xgb,
    y=y_train, 
    eval_set=[(X_val_xgb, y_val)], 
    verbose=10
)

y_pred_xgb = xgb_reg.predict(X_test_xgb)
print(f"\nFull XGBoost Baseline Metrics:")
print(f"Test MAE: {mean_absolute_error(y_test, y_pred_xgb):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_xgb)):.4f}")
print(f"Test R2: {r2_score(y_test, y_pred_xgb):.4f}")

### Tabular-only XGBoost model

In [ ]:
xgb_reg.fit(
    X=X_tab_train,
    y=y_train, 
    eval_set=[(X_tab_val, y_val)], 
    verbose=10
)

y_pred_tab_xgb = xgb_reg.predict(X_tab_test)
print(f"\nXGBoost Baseline Metrics:")
print(f"Test MAE: {mean_absolute_error(y_test, y_pred_tab_xgb):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_tab_xgb)):.4f}")
print(f"Test R2: {r2_score(y_test, y_pred_tab_xgb):.4f}")

### Embedding-only Linear Baseline

In [ ]:
class EmbeddingLinearModel(nn.Module):
    def __init__(self, text_dim):
        super(EmbeddingLinearModel, self).__init__()
        self.linear = nn.Linear(text_dim, 1)

    def forward(self, text_embeddings, _):
        out = self.linear(text_embeddings)
        return out.squeeze(-1)

linear_model = EmbeddingLinearModel(text_dim=len(embedding_cols)).to(DEVICE)
linear_optimizer = torch.optim.SGD(linear_model.parameters(), lr=0.001)
linear_criterion = nn.HuberLoss(delta=3.0)
best_linear_model = None

best_linear_loss = float('inf')
for epoch in range(EPOCHS):
    linear_model.train()
    train_loss = 0
    for texts, tabs, labels in train_loader:
        texts, labels = texts.to(DEVICE), labels.to(DEVICE)
        linear_optimizer.zero_grad()
        outputs = linear_model(texts, None)
        loss = linear_criterion(outputs, labels)
        loss.backward()
        linear_optimizer.step()
        train_loss += loss.item() * texts.size(0)
    train_loss /= len(train_loader.dataset)
    
    linear_model.eval()
    val_loss = 0
    with torch.no_grad():
        for texts, tabs, labels in val_loader:
            texts, labels = texts.to(DEVICE), labels.to(DEVICE)
            outputs = linear_model(texts, None)
            loss = linear_criterion(outputs, labels)
            val_loss += loss.item() * texts.size(0)
    val_loss /= len(val_loader.dataset)

    if val_loss < best_linear_loss:
        best_linear_loss = val_loss
        # TODO: save best linear baseline in memory (do i need deepcopy for that?)
        best_linear_model = linear_model

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

print(f"Embedding Linear Baseline training complete. Best Val Loss: {best_linear_loss:.4f}")

# test linear embedding model
best_linear_model.eval()

linear_preds = []
with torch.no_grad():
    for texts, tabs, labels in test_loader:
        texts = texts.to(DEVICE)
        outputs = best_linear_model(texts, None)
        linear_preds.extend(outputs.cpu().numpy())
y_pred_linear = np.array(linear_preds)

## Train late fusion model

In [ ]:
model = MTGFusionModel(
    text_dim=len(embedding_cols), 
    tabular_dim=X_tab_train.shape[1]
).to(DEVICE)

criterion = torch.nn.HuberLoss(delta=3.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.1)

In [ ]:
best_loss = float('inf')

for epoch in range(EPOCHS):
    # train
    model.train()
    train_loss = 0
    for texts, tabs, labels in train_loader:
        texts, tabs, labels = texts.to(DEVICE), tabs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(texts, tabs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * texts.size(0)
        
    train_loss /= len(train_loader.dataset)
    
    # validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for texts, tabs, labels in val_loader:
            texts, tabs, labels = texts.to(DEVICE), tabs.to(DEVICE), labels.to(DEVICE)
            outputs = model(texts, tabs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * texts.size(0)
    
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "best_fusion_model.pth")
        
print(f"Training complete. Best Val Loss: {best_loss:.4f}")

## Test model

In [ ]:
best_fusion_model = MTGFusionModel(
    text_dim=len(embedding_cols), 
    tabular_dim=X_tab_train.shape[1]
).to(DEVICE)
best_fusion_model.load_state_dict(torch.load("best_fusion_model.pth"))
best_fusion_model.eval()

fusion_preds = []
with torch.no_grad():
    for texts, tabs, labels in test_loader:
        texts, tabs = texts.to(DEVICE), tabs.to(DEVICE)
        outputs = best_fusion_model(texts, tabs)
        fusion_preds.extend(outputs.cpu().numpy())
        
y_pred_fusion = np.array(fusion_preds)

## Compare results

In [ ]:
models_metrics = {}
preds = {
    "Full XGBoost": y_pred_xgb,
    "Tabular XGBoost": y_pred_tab_xgb,
    "Embedding Baseline": y_pred_linear,
    "Late Fusion": y_pred_fusion
}

for name, y_pred in preds.items():
    models_metrics[name] = {
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred)
    }
    print(f"{name} Metrics:")
    print(f"  Test MAE: {models_metrics[name]['MAE']:.4f}")
    print(f"  Test RMSE: {models_metrics[name]['RMSE']:.4f}")
    print(f"  Test R2: {models_metrics[name]['R2']:.4f}")

In [ ]:
xgb_r2 = models_metrics["Full XGBoost"]["R2"]
xgb_mae = models_metrics["Full XGBoost"]["MAE"]
xgb_rmse = models_metrics["Full XGBoost"]["RMSE"]
fusion_r2 = models_metrics["Late Fusion"]["R2"]
fusion_mae = models_metrics["Late Fusion"]["MAE"]
fusion_rmse = models_metrics["Late Fusion"]["RMSE"]


plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred_xgb, alpha=0.3, label=f'XGBoost (R2={xgb_r2:.2f})', color='tab:blue')
plt.scatter(y_test, y_pred_fusion, alpha=0.3, label=f'Late Fusion (R2={fusion_r2:.2f})', color='tab:orange')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel('Actual Power Score')
plt.ylabel('Predicted Power Score')
plt.title('Prediction vs Actual')
plt.legend()

# 2. Plot Error Distribution
plt.subplot(1, 2, 2)
sns.kdeplot(y_test - y_pred_xgb, label='XGBoost Residuals', fill=True, color='tab:blue')
sns.kdeplot(y_test - y_pred_fusion, label='Late Fusion Residuals', fill=True, color='tab:orange')
plt.axvline(x=0, color='red', linestyle='--')
plt.xlabel('Residual (Actual - Predicted)')
plt.title('Residual Distribution')
plt.legend()

plt.tight_layout()
plt.show()

# 3. Metrics Comparison Bar Chart
metrics_df = pd.DataFrame({
    'Model': ['XGBoost', 'Late Fusion', 'XGBoost', 'Late Fusion'],
    'Metric': ['MAE', 'MAE', 'RMSE', 'RMSE'],
    'Value': [xgb_mae, fusion_mae, xgb_rmse, fusion_rmse]
})

plt.figure(figsize=(8, 5))
sns.barplot(x='Metric', y='Value', hue='Model', data=metrics_df)
plt.title('Model Comparison: MAE and RMSE')
plt.show()